In [9]:
import xml.etree.ElementTree as ET
import os
import tensorflow as tf
def parse_voc_annotations(annotation_dir, image_dir, class_map):
    image_paths = []
    bbox_data =[]
    class_labels =[]
    for xml_file in sorted(os.listdir(annotation_dir)):
        if not xml_file.endswith(".xml"):
            continue
        tree = ET.parse(os.path.join(annotation_dir,xml_file))
        root= tree.getroot()
        image_file_name = root.find("filename").text
        path = os.path.join(image_dir,image_file_name)

        size = root.find("size")
        img_width = int(size.find("width").text)
        img_height = int(size.find("height").text)

        obj = root.find("object")
        if obj is not None:
            class_name = obj.find("name").text
            if class_name not in class_map:
                continue
            class_id = class_map[class_name]

            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text) / img_width
            ymin = float(bndbox.find("ymin").text) / img_height
            xmax = float(bndbox.find("xmax").text) / img_width
            ymax = float(bndbox.find("ymax").text) / img_height
            image_paths.append(path)
            bbox_data.append([xmin, ymin, xmax, ymax])
            class_labels.append(class_id)
    return image_paths, bbox_data, class_labels


In [ ]:
image_dir = "../data/images/"
annotation_dir = "../data/Annotations/"
class_map = {"Cat": 0, "Dog": 1}

image_paths, bbox_data, class_labels = parse_voc_annotations(
    annotation_dir=annotation_dir,
    image_dir=image_dir,
    class_map=class_map
)

In [ ]:
image_paths = tf.constant(image_paths)
bbox_data = tf.constant(bbox_data, dtype=tf.float32)
class_labels = tf.constant(class_labels, dtype=tf.int32)

def load_and_preprocess_image(path, bbox, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [128, 128])
    image = image / 255.0  
    return image, {"class_output": label, "box_output": bbox}


dataset = tf.data.Dataset.from_tensor_slices((image_paths, bbox_data, class_labels))
dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)


dataset = dataset.shuffle(buffer_size=1024).batch(32).prefetch(tf.data.AUTOTUNE)

DATASET_SIZE = len(image_paths)
train_size = int(0.8 * DATASET_SIZE)

train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

inputs = Input(shape=(128, 128, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)

box_output = Dense(4, activation='sigmoid', name='box_output')(x) 

class_output = Dense(len(class_map), activation='softmax', name='class_output')(x)

model = Model(inputs=inputs, outputs=[class_output, box_output])


model.compile(optimizer='adam',
              loss={
                  'class_output': 'sparse_categorical_crossentropy',
                  'box_output': 'mae' 
              },
              metrics={
                  'class_output': 'accuracy',
                  'box_output': 'mae'
              })

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_1… │ (None, 4, 4,      │  2,257,984 │ input_layer_5[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 2)         │      2,562 │ global_average_p… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ box_output (Dense)  │ (None, 4)         │      5,124 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.fit(train_ds,
            epochs=10,
            validation_data=val_ds,
            callbacks =[tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, mode="max"),
                        tf.keras.callbacks.ModelCheckpoint("../trained_model/bestmodel.keras", save_best_only=True, monitor="val_accuracy", mode="max", verbose=1)],
            verbose=1
                    )


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 752ms/step - box_output_loss: 0.1052 - box_output_mae: 0.1050 - class_output_accuracy: 1.0000 - class_output_loss: 0.0356 - loss: 0.1402
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 500ms/step - box_output_loss: 0.1059 - box_output_mae: 0.1077 - class_output_accuracy: 1.0000 - class_output_loss: 0.0307 - loss: 0.1376
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 526ms/step - box_output_loss: 0.1034 - box_output_mae: 0.1029 - class_output_accuracy: 1.0000 - class_output_loss: 0.0267 - loss: 0.1290
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 572ms/step - box_output_loss: 0.1034 - box_output_mae: 0.1053 - class_output_accuracy: 1.0000 - class_output_loss: 0.0245 - loss: 0.1300
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 610ms/step - box_output_loss: 0.1018 - box_output_mae: 0.1002 - class_output_accuracy: 1.0000 - class_output_loss: 0.0208 - loss: 0.1203
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 558ms/step - box_output_loss: 0.0891 - box_output_mae: 0.0884 - class_output_accu